# ProductIQ Phase 2 — Step 2.10: PostgreSQL Product Catalog

PostgreSQL is the serving boundary after Phase 2.9 processed Parquet.

This notebook documents configuration, schema, chunked loading, idempotent upsert, validation, and repository access.

## 1. Architecture

- **Configuration:** `DATABASE_URL` → `Settings`
- **Engine / sessions:** lazy SQLAlchemy engine + `session_scope()`
- **ORM:** `Product` (`products` table, 22 columns)
- **Loader:** chunked Parquet → PostgreSQL upsert (`ProductCatalogLoader`)
- **Pipeline:** `load_processed_catalog()`
- **Validation:** `validate_product_catalog(engine)` queries PostgreSQL directly
- **Repository:** `ProductRepository` for basic retrieval/filtering

## 2. Configure DATABASE_URL

Set `DATABASE_URL` in the environment or `.env`. Use `settings.redacted_database_url` for logs.

In [ ]:
from productiq.config import get_settings

settings = get_settings()
print("Redacted DATABASE_URL:", settings.redacted_database_url)

## 3. Product schema (22 columns)

Aligned with `resources/processed/product_catalog.parquet` schema version `1.0.0`.

In [ ]:
from productiq.database import (
    CATALOG_COLUMN_ORDER,
    CATALOG_NULLABLE_ATTRIBUTE_COLUMNS,
    CATALOG_REQUIRED_COLUMNS,
    PRODUCTS_TABLE_NAME,
    Product,
)

print("Table:", PRODUCTS_TABLE_NAME)
print("Columns:", len(CATALOG_COLUMN_ORDER))
print("Required:", len(CATALOG_REQUIRED_COLUMNS))
print("Nullable attributes:", len(CATALOG_NULLABLE_ATTRIBUTE_COLUMNS))

## 4. Create table (development)

Requires a reachable PostgreSQL instance.

In [ ]:
from productiq.database import create_engine_from_settings, create_product_catalog_tables

engine = create_engine_from_settings(settings)
create_product_catalog_tables(engine)
print("products table ensured")

## 5. Loader architecture

- Reads processed Parquet with PyArrow batches
- Default chunk size: `10_000` rows
- Uses PostgreSQL `INSERT ... ON CONFLICT (product_id) DO UPDATE`
- Commits once per chunk (controlled transactions)
- Does not mutate the source Parquet file

## 6. End-to-end pipeline (explicit step)

**Do not run automatically.** Execute only when PostgreSQL is configured.

Full AJIO load expects `367,172` rows from `resources/processed/product_catalog.parquet`.

In [ ]:
# from pathlib import Path
# from productiq.database import load_processed_catalog
#
# report = load_processed_catalog(Path("resources/processed/product_catalog.parquet"), engine=engine)
# print(report.summary())

## 7. Validate catalog in PostgreSQL

In [ ]:
# from productiq.database import validate_product_catalog
#
# validation = validate_product_catalog(engine, expected_row_count=367_172)
# print(validation.summary())

## 8. Repository examples

In [ ]:
# from productiq.database import ProductRepository, session_scope
#
# with session_scope() as session:
#     repo = ProductRepository(session)
#     print("Count:", repo.count())
#     print("Sample:", repo.get_by_id("123456789001"))

## 9. Phase 2.10 summary

- Catalog table, constraints, and indexes match the processed dataset contract.
- Loader is chunked, idempotent, and validated against PostgreSQL after load.
- Repository provides paginated retrieval for future search phases.